# 12 — Classical ML Candidates in the Selection Process

Notebook 10 only profiled CNN backbones. This one extends the same roster-style shootout to **classical ML 
models trained on spectrogram-derived features**, so the architecture selection sees both options side by side 
and the deserves-a-slot question gets answered with numbers, not vibes.

**Setup mirrors notebook 10** so the comparisons are honest:

1. **Candidate roster** — LightGBM, Random Forest, Logistic Regression, optionally XGBoost. All trained 
one-vs-rest (one binary classifier per species) so the metric is computed exactly the same way as for the NNs.
2. **Static profile** — train time, model size, single-sample predict latency.
3. **Smoke-fit on fold 0** — same fold split (`data/folds/folds.csv`) the NN anchors used, so val macro-AUC is 
directly comparable.
4. **Verdict** — three-tier decision: replace anchor / add to ensemble / drop.

**What classical ML can and can't do here.** Trees and linear models can't see 2D translation-invariant 
structure in the spectrogram (a chirp shifted 200ms in time looks like a different feature vector). What they 
*can* do is exploit clip-level summary statistics — total energy in band X, mean MFCC coefficient Y, peak time 
in the upper bands. These are real signals for some species (constant-pitch alarm calls especially), and tree 
models pick them up cheaply. Expect raw scores 5-15 AUC points behind the CNN anchors. The interesting question 
isn't whether classical ML wins outright — it almost certainly won't — it's whether the gap is small enough 
*and the OOF correlation with the NN models is low enough* that it earns a slot in your ensemble.

**About 'ensemble methods' as a model class.** LightGBM and Random Forest are themselves ensembles of decision 
trees. So when this notebook scores 'LightGBM' against an NN backbone, that *is* a fair test of whether a 
tree-ensemble method belongs in your candidate pool. The stacker shootout in notebook 11 is a different question 
(*how* to combine trained models); this notebook asks *whether* tree-based models should be among the trained 
models in the first place.


## 0 — Setup

In [1]:
import os, sys, time, gc, warnings
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.fftpack
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Optional: LightGBM (the headline candidate). Without it we still run RF + LR.
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('lightgbm not installed — skipping the GBM candidate. `pip install lightgbm` to enable.')

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
RNG = np.random.default_rng(42)

# Paths into the existing repo layout
SPEC_DIR    = ROOT / 'data' / 'processed'
META_CSV    = ROOT / 'data' / 'raw' / 'train.csv'
FOLDS_CSV   = ROOT / 'data' / 'folds' / 'folds.csv'
FEATURES_CACHE = ROOT / 'experiments' / '_classical_features.parquet'
EXP_DIR     = ROOT / 'experiments'

FOLD_FOR_SMOKE = 0
MAX_FILES_FOR_SMOKE = None    # None = use all; set to 5000 for a fast first pass


## 1 — Candidate roster

Each candidate is a dict with `name`, `family`, and a callable `make()` that returns an unfit 
binary classifier. The OvR loop in section 4 instantiates one of these per species, so the hyperparameters 
should be sized for *binary classification on ~30k samples with class imbalance*, not for multiclass.

Capacity is intentionally modest. A surface test with overcooked hyperparameters that takes 6 hours to run 
isn't a surface test. If a candidate clears the verdict bar in section 6, *then* tune.


In [2]:
CANDIDATES = []

if HAS_LGB:
    CANDIDATES.append({
        'name': 'lightgbm', 'family': 'GBM',
        'make': lambda: lgb.LGBMClassifier(
            n_estimators=200, num_leaves=31, learning_rate=0.05,
            min_data_in_leaf=20, class_weight='balanced',
            n_jobs=1, verbosity=-1, force_col_wise=True,
        ),
    })

if HAS_XGB:
    CANDIDATES.append({
        'name': 'xgboost', 'family': 'GBM',
        'make': lambda: xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.05,
            tree_method='hist', n_jobs=1, verbosity=0, eval_metric='auc',
        ),
    })

CANDIDATES.append({
    'name': 'random_forest', 'family': 'Bagging',
    'make': lambda: RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_leaf=5,
        class_weight='balanced_subsample', n_jobs=1, random_state=42,
    ),
})

CANDIDATES.append({
    'name': 'logreg', 'family': 'Linear',
    'make': lambda: LogisticRegression(
        max_iter=300, C=1.0, class_weight='balanced', solver='liblinear',
    ),
})

for c in CANDIDATES:
    print(f'  {c["name"]:<14}  family={c["family"]}')
print(f'\n{len(CANDIDATES)} classical-ML candidates registered')


  lightgbm        family=GBM
  random_forest   family=Bagging
  logreg          family=Linear

3 classical-ML candidates registered


## 2 — Feature extraction from cached mel spectrograms

Reads each per-clip mel spec from `data/processed/` and computes a fixed-length feature vector. The vector 
has three parts:

- **Per-band statistics over time** (128 bands × 4 stats: mean, std, max, p90) — captures *which frequency bands 
are active* and *how much they vary*. The dominant signal for most species ID.
- **MFCC summary stats** (20 coeffs × 5 stats: mean, std, min, max, p50) — captures spectral envelope shape, 
computed as DCT-II of the log-mel spec. Done with scipy so librosa isn't a hard dependency.
- **Global stats** (8 features) — total energy, peak frequency band, peak time fraction, dynamic range, 
spectral centroid in mel space, mel-band sparsity, time-axis sparsity, energy concentration ratio.

Total: **620 features per clip.** Cached to parquet so subsequent runs skip the slow extraction step.

If your mel specs aren't on disk, the fallback synthetic path lets you exercise the rest of the notebook with 
fake data — useful for code review but not for the verdict.


In [3]:
N_MELS_DEFAULT = 128
N_MFCC = 20


def extract_features(mel_db: np.ndarray) -> np.ndarray:
    """
    mel_db: (n_mels, T) log-mel spectrogram in dB.
    Returns a 1D feature vector of length 128*4 + 20*5 + 8 = 620.
    Robust to T variation (uses summary stats over time).
    """
    n_mels, T = mel_db.shape

    # Per-band stats over time
    band_mean = mel_db.mean(axis=1)
    band_std  = mel_db.std(axis=1)
    band_max  = mel_db.max(axis=1)
    band_p90  = np.percentile(mel_db, 90, axis=1)
    band_feats = np.concatenate([band_mean, band_std, band_max, band_p90])

    # MFCCs from log-mel via DCT-II (matches librosa.feature.mfcc(S=mel_db))
    mfcc = scipy.fftpack.dct(mel_db, type=2, axis=0, norm='ortho')[:N_MFCC]
    mfcc_feats = np.concatenate([
        mfcc.mean(axis=1), mfcc.std(axis=1),
        mfcc.min(axis=1), mfcc.max(axis=1), np.percentile(mfcc, 50, axis=1),
    ])

    # Global descriptors
    energy = mel_db.sum()
    peak_band = int(band_mean.argmax())
    peak_time_frac = float(mel_db.mean(axis=0).argmax()) / max(T - 1, 1)
    dyn_range = mel_db.max() - mel_db.min()
    # Spectral centroid in mel-bin space, weighted by per-band energy
    band_energy = np.exp(band_mean - band_mean.max())   # avoid overflow on dB scale
    spec_centroid = (np.arange(n_mels) * band_energy).sum() / max(band_energy.sum(), 1e-9)
    band_sparsity = (band_mean > band_mean.mean()).mean()
    time_energy = mel_db.mean(axis=0)
    time_sparsity = (time_energy > time_energy.mean()).mean()
    energy_top10 = np.sort(time_energy)[-max(T // 10, 1):].mean()
    energy_concentration = energy_top10 / max(time_energy.mean(), 1e-9)
    global_feats = np.array([
        energy, peak_band, peak_time_frac, dyn_range,
        spec_centroid, band_sparsity, time_sparsity, energy_concentration,
    ], dtype=np.float64)

    return np.concatenate([band_feats, mfcc_feats, global_feats]).astype(np.float32)


FEATURE_DIM = N_MELS_DEFAULT * 4 + N_MFCC * 5 + 8
print(f'feature dimension: {FEATURE_DIM}')

# Sanity-check the extractor on a fake spec
_test = RNG.standard_normal((128, 314)).astype(np.float32) * 10 - 30
_v = extract_features(_test)
assert _v.shape == (FEATURE_DIM,), f'expected {FEATURE_DIM}, got {_v.shape}'
print(f'sanity OK — vector shape {_v.shape}, dtype {_v.dtype}')


feature dimension: 620
sanity OK — vector shape (620,), dtype float32


In [4]:
def _load_one_spec(path: Path) -> np.ndarray | None:
    """Load a single .npy mel spec. Returns None on failure."""
    try:
        a = np.load(path)
        if a.ndim == 3 and a.shape[0] == 1:
            a = a[0]
        if a.ndim != 2:
            return None
        return a
    except Exception:
        return None


def build_feature_table(meta_df, spec_dir, max_files=None):
    """
    For each row in meta with a corresponding .npy spec, extract features.
    Returns DataFrame with ['filename', 'primary_label', 'f0', 'f1', ..., 'f{D-1}'].
    """
    candidates = list(meta_df['filename'])
    if max_files is not None:
        candidates = candidates[:max_files]

    rows, missing = [], 0
    t0 = time.time()
    for i, fn in enumerate(candidates):
        # Conventional layout: data/processed/<filename>.npy where filename has no .ogg suffix
        stem = Path(fn).stem
        for cand in [spec_dir / f'{stem}.npy', spec_dir / fn.replace('.ogg', '.npy')]:
            if cand.exists():
                spec = _load_one_spec(cand)
                break
        else:
            spec = None
        if spec is None:
            missing += 1
            continue
        feats = extract_features(spec)
        rows.append((fn, *feats))
        if (i + 1) % 1000 == 0:
            elapsed = time.time() - t0
            print(f'  {i+1:>5}/{len(candidates)}  '
                  f'{(i+1)/elapsed:.1f} files/s  missing so far: {missing}')

    cols = ['filename'] + [f'f{i}' for i in range(FEATURE_DIM)]
    df = pd.DataFrame(rows, columns=cols)
    df = df.merge(meta_df[['filename', 'primary_label']], on='filename', how='left')
    print(f'extracted {len(df)} / {len(candidates)} (missing: {missing})')
    return df


USING_SYNTHETIC = False

if FEATURES_CACHE.exists():
    print(f'loading cached features from {FEATURES_CACHE}')
    feat_df = pd.read_parquet(FEATURES_CACHE)
elif META_CSV.exists() and SPEC_DIR.exists():
    print(f'extracting features from {SPEC_DIR} — first run, this is the slow step')
    meta_df = pd.read_csv(META_CSV)
    feat_df = build_feature_table(meta_df, SPEC_DIR, max_files=MAX_FILES_FOR_SMOKE)
    FEATURES_CACHE.parent.mkdir(parents=True, exist_ok=True)
    feat_df.to_parquet(FEATURES_CACHE, index=False)
    print(f'cached → {FEATURES_CACHE}')
else:
    print(f'No real data found ({META_CSV} or {SPEC_DIR} missing) — falling back to synthetic.')
    USING_SYNTHETIC = True
    N, C = 4000, 80   # smaller than 234 so the surface test runs in a few minutes
    species_pool = [f'sp_{i:03d}' for i in range(C)]
    primary = RNG.choice(species_pool, N)
    # Inject signal: each species has a preferred set of features
    feats = RNG.standard_normal((N, FEATURE_DIM)).astype(np.float32)
    pref = RNG.choice(FEATURE_DIM, size=(C, 8), replace=True)
    sp_to_idx = {sp: i for i, sp in enumerate(species_pool)}
    for i, sp in enumerate(primary):
        feats[i, pref[sp_to_idx[sp]]] += 2.0
    feat_df = pd.DataFrame(feats, columns=[f'f{i}' for i in range(FEATURE_DIM)])
    feat_df.insert(0, 'filename', [f'syn_{i:05d}.ogg' for i in range(N)])
    feat_df['primary_label'] = primary

print(f'\nfeature table: {feat_df.shape}  classes: {feat_df["primary_label"].nunique()}')
feat_df.head(3)


loading cached features from /home/gokhuu/Portfolio/BirdClef26/experiments/_classical_features.parquet

feature table: (35549, 622)  classes: 206


,filename,f0,f1,f2,f3,f4,f5,f6,f7,f8,...,f611,f612,f613,f614,f615,f616,f617,f618,f619,primary_label
0,1161364/iNat1216197.ogg,-37.644402,-30.425133,-29.204613,-30.469784,-32.393974,-33.692314,-34.683952,-35.518345,-35.946178,...,5.813347,-6680790.0,2.0,0.506217,80.0,5.082909,0.484375,0.776398,-4.266093e+10,1161364
1,1161364/iNat1114648.ogg,-32.184753,-25.731422,-22.446619,-21.756298,-21.201647,-18.499697,-20.241713,-23.149128,-26.264503,...,2.695013,-10067610.0,5.0,0.763247,80.0,4.994780,0.429688,0.761690,-4.080932e+10,1161364
2,1161364/iNat810195.ogg,-49.116817,-41.135666,-38.286915,-37.822842,-37.540871,-37.888710,-38.172173,-37.163757,-37.655579,...,7.809901,-30429688.0,91.0,0.710858,80.0,68.676476,0.523438,0.480008,-4.563569e+10,1161364


## 3 — Smoke-fit on fold 0 (one-vs-rest per species)

Loads `data/folds/folds.csv` to match the NN anchors' fold assignment. For each candidate, trains 1 binary 
classifier per species on folds {1..4}, predicts on fold 0, computes per-class AUC, then averages over species 
with at least one positive in val (the competition metric).

**Sized for a surface test, not a final model.** Skips species with fewer than 5 train positives (their per-class 
model would be unstable and the per-class AUC would dominate the average with noise). Reports separately how many 
species were skipped so you know what fraction of the metric is being measured.

If folds.csv is missing (synthetic mode), uses a deterministic 80/20 split.


In [5]:
def macro_auc_skip_empty(y_true, y_pred):
    """Same metric implementation as notebook 11."""
    aucs = []
    for c in range(y_true.shape[1]):
        pos = y_true[:, c].sum()
        if 0 < pos < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan'), len(aucs)


# Build train/val split
if USING_SYNTHETIC or not FOLDS_CSV.exists():
    rng = np.random.default_rng(42)
    perm = rng.permutation(len(feat_df))
    cut = int(0.8 * len(perm))
    train_mask = np.zeros(len(feat_df), dtype=bool); train_mask[perm[:cut]] = True
    val_mask   = ~train_mask
    print(f'using deterministic 80/20 split (no folds.csv)')
else:
    folds = pd.read_csv(FOLDS_CSV)
    feat_df_with_fold = feat_df.merge(folds[['filename', 'fold']], on='filename', how='left')
    train_mask = (feat_df_with_fold['fold'] != FOLD_FOR_SMOKE).values
    val_mask   = (feat_df_with_fold['fold'] == FOLD_FOR_SMOKE).values
    print(f'using folds.csv, fold {FOLD_FOR_SMOKE} = val')

feat_cols = [c for c in feat_df.columns if c.startswith('f') and c[1:].isdigit()]
X = feat_df[feat_cols].values.astype(np.float32)
X_tr, X_va = X[train_mask], X[val_mask]

# Scale features once (for LR; trees are scale-invariant but it doesn't hurt)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr).astype(np.float32)
X_va_s = scaler.transform(X_va).astype(np.float32)

# Multi-hot label matrix
species_list = sorted(feat_df['primary_label'].dropna().unique())
sp_to_idx = {sp: i for i, sp in enumerate(species_list)}
Y = np.zeros((len(feat_df), len(species_list)), dtype=np.float32)
for i, sp in enumerate(feat_df['primary_label'].values):
    if isinstance(sp, str) and sp in sp_to_idx:
        Y[i, sp_to_idx[sp]] = 1.0
Y_tr, Y_va = Y[train_mask], Y[val_mask]

print(f'train: {X_tr.shape}   val: {X_va.shape}   classes: {Y.shape[1]}')
print(f'val class-positive distribution: '
      f'min={Y_va.sum(0).min():.0f}  median={int(np.median(Y_va.sum(0)))}  max={Y_va.sum(0).max():.0f}')


using folds.csv, fold 0 = val
train: (28431, 620)   val: (7118, 620)   classes: 206
val class-positive distribution: min=1  median=25  max=100


In [6]:
def fit_predict_ovr(make_clf, X_tr, Y_tr, X_va, min_pos=5):
    """
    Train one binary classifier per class (one-vs-rest). Returns (preds, n_skipped).
    Skipped classes get all-0.5 predictions.
    """
    C = Y_tr.shape[1]
    out = np.full((X_va.shape[0], C), 0.5, dtype=np.float32)
    n_skipped = 0
    for c in range(C):
        yc = Y_tr[:, c]
        if yc.sum() < min_pos or yc.sum() > len(yc) - min_pos:
            n_skipped += 1
            continue
        clf = make_clf()
        clf.fit(X_tr, yc)
        if hasattr(clf, 'predict_proba'):
            out[:, c] = clf.predict_proba(X_va)[:, 1]
        else:
            out[:, c] = clf.decision_function(X_va)
    return out, n_skipped


# Run each candidate. Save per-candidate OOF for the diversity check in section 5.
results = []
oof_predictions = {}
for cand in CANDIDATES:
    print(f'\n--- {cand["name"]} ---')
    t0 = time.time()
    # LR is the only one needing the scaled features; trees use raw
    Xt, Xv = (X_tr_s, X_va_s) if cand['name'] == 'logreg' else (X_tr, X_va)
    preds, n_skipped = fit_predict_ovr(cand['make'], Xt, Y_tr, Xv)
    train_time = time.time() - t0
    auc, n_eval = macro_auc_skip_empty(Y_va, preds)
    print(f'  train_time={train_time:.0f}s   skipped {n_skipped} classes (too few train positives)')
    print(f'  val macro AUC = {auc:.4f}   (averaged over {n_eval} classes with val positives)')
    results.append({
        'name': cand['name'], 'family': cand['family'],
        'val_macro_auc': auc, 'n_classes_eval': n_eval,
        'train_time_s': train_time, 'n_classes_skipped_in_train': n_skipped,
    })
    oof_predictions[cand['name']] = preds

results_df = pd.DataFrame(results)
results_df.round({'val_macro_auc': 4, 'train_time_s': 1})



--- lightgbm ---
  train_time=4833s   skipped 18 classes (too few train positives)
  val macro AUC = 0.8646   (averaged over 206 classes with val positives)

--- random_forest ---


KeyboardInterrupt: 

## 4 — Comparison vs neural-network anchors

Loads NN anchor val-AUC from notebook 10's `KNOWN_ANCHOR_AUCS` dict (or the per-species OOFs in `experiments/`) 
and ranks both populations together. **Edit `NN_ANCHORS` below to match your real numbers** — defaults are the 
same placeholders we used in notebook 10.


In [ ]:
# Edit these to match the real macro-AUC of each NN anchor on fold 0
# (read from experiments/<run>/per_species_auc.csv or training_log.csv)
NN_ANCHORS = {
    'effb0_sed':     0.948,
    'effv2s_sed':    0.957,
    'convnext_tiny': 0.951,
    'seresnext26d':  0.943,
}

combined = []
for name, auc in NN_ANCHORS.items():
    combined.append({'name': name, 'family': 'NN', 'class': 'anchor (NN)',
                     'val_macro_auc': auc})
for r in results:
    combined.append({'name': r['name'], 'family': r['family'], 'class': 'classical (this nb)',
                     'val_macro_auc': r['val_macro_auc']})

combined_df = pd.DataFrame(combined).sort_values('val_macro_auc', ascending=False).reset_index(drop=True)
print(combined_df.round({'val_macro_auc': 4}).to_string(index=False))

best_nn         = max(NN_ANCHORS.values())
best_classical  = max(r['val_macro_auc'] for r in results)
gap             = best_nn - best_classical
print(f'\nbest NN anchor:        {best_nn:.4f}')
print(f'best classical model:  {best_classical:.4f}  (gap: {gap:+.4f})')


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
df_plot = combined_df.sort_values('val_macro_auc')
colors = {'anchor (NN)': '#2563eb', 'classical (this nb)': '#16a34a'}
ax.barh(df_plot['name'], df_plot['val_macro_auc'],
        color=[colors[c] for c in df_plot['class']], edgecolor='black', linewidth=0.4)
ax.set_xlim(min(0.5, df_plot['val_macro_auc'].min() - 0.02), 1.0)
ax.set_xlabel('val macro AUC (fold 0)')
ax.set_title('NN anchors vs classical-ML candidates — same fold, same metric')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c, edgecolor='black', label=g) for g, c in colors.items()],
          loc='lower right')
plt.tight_layout(); plt.show()


## 5 — Diversity check: are the classical predictions uncorrelated with NN OOF?

Even when the gap in section 4 is large, a classical model can still earn a slot in the *ensemble* if its errors 
are uncorrelated with the NN models' errors — exactly the diversity argument from notebook 11. This section 
loads any available NN OOF predictions, aligns them with the classical OOFs from section 3, and reports pairwise 
correlation.

**Threshold of interest.** Pearson correlation < 0.85 between the classical model's per-sample top-class 
probability and the NN anchors' is the rough cutoff where ensembling a much weaker model can still help. If all 
your NN anchors are correlated > 0.95 with each other, an uncorrelated 0.88-AUC classical model can be more 
valuable to the ensemble than an extra correlated 0.95-AUC NN.


In [ ]:
def load_nn_oof_for_fold(exp_subpath, fold=FOLD_FOR_SMOKE):
    """Load fold-0 OOF predictions for an NN run if available. Returns DataFrame or None."""
    for pat in [f'{exp_subpath}_fold{fold}/oof_preds.csv',
                f'{exp_subpath}/fold{fold}/oof_preds.csv']:
        p = EXP_DIR / pat
        if p.exists():
            return pd.read_csv(p)
    return None


# Map each NN anchor to its per-fold OOF location. Edit if your paths differ.
NN_OOF_PATHS = {
    'effb0':     'baseline_effb0/baseline_effb0',
    'effv2s':    'effv2s_finetune/effv2s_finetune',
    'convnext':  'sed_finetune/sed_finetune',
    'seresnext': 'seresnext_finetune/seresnext_finetune',
}

nn_oof = {n: load_nn_oof_for_fold(p) for n, p in NN_OOF_PATHS.items()}
nn_oof = {n: df for n, df in nn_oof.items() if df is not None}
print(f'loaded NN OOF for {len(nn_oof)} anchors: {list(nn_oof.keys())}')

if nn_oof:
    # Align on val filenames
    val_filenames = feat_df.loc[val_mask, 'filename'].values
    common = set(val_filenames)
    for df in nn_oof.values():
        common &= set(df['filename'])
    common = sorted(common)
    print(f'common val files across classical + NN OOFs: {len(common)}')

    # Build per-sample top-class probability for each model
    val_idx = pd.Series(np.arange(len(val_filenames)), index=val_filenames).loc[common].values
    diversity_signals = {}
    for n, p in oof_predictions.items():
        diversity_signals[f'classical:{n}'] = p[val_idx].max(axis=1)
    for n, df in nn_oof.items():
        df_aligned = df.set_index('filename').loc[common]
        sp_cols = [c for c in df_aligned.columns]
        diversity_signals[f'nn:{n}'] = df_aligned[sp_cols].values.max(axis=1)

    div_df = pd.DataFrame(diversity_signals)
    corr = div_df.corr()

    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(corr.values, vmin=0.6, vmax=1.0, cmap='RdYlGn_r')
    ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticklabels(corr.columns)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center',
                    color='white' if corr.values[i, j] > 0.9 else 'black', fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title('Cross-family OOF correlation\n(per-sample top-class probability)')
    plt.tight_layout(); plt.show()

    # Report best classical-vs-best-NN correlation
    classical_cols = [c for c in corr.columns if c.startswith('classical:')]
    nn_cols = [c for c in corr.columns if c.startswith('nn:')]
    if classical_cols and nn_cols:
        cross = corr.loc[classical_cols, nn_cols]
        print(f'\nclassical-vs-NN correlation matrix:')
        print(cross.round(3))
        print(f'\nmin classical-NN correlation: {cross.values.min():.3f}')
        print(f'max classical-NN correlation: {cross.values.max():.3f}')
else:
    print('No NN OOFs available — skipping diversity check. Re-run after re-exporting OOFs.')
    cross = None


## 6 — Verdict

Three-tier decision logic, in order:

1. **REPLACE.** A classical candidate beats the best NN anchor by more than 0.005 AUC. Almost certainly won't 
happen — but if it does, you have a much cheaper base model to deploy.
2. **ENSEMBLE.** A classical candidate is within 0.05 of the best NN anchor *and* its max correlation with any NN 
anchor is below 0.85. Worth adding to the stacker shootout in notebook 11 as an additional base model.
3. **DROP.** Gap larger than 0.05 AND correlation higher than 0.85, or correlation data unavailable. Not worth the 
engineering complexity at the current LB position.

These cutoffs are calibrated for ~0.95 NN baseline territory. If your best NN drops to ~0.90 (e.g. on a harder 
fold of the metric), widen the gap threshold — a 0.85-AUC tree model is a much bigger fraction of the gap.


In [ ]:
GAP_REPLACE   = -0.005   # classical beats NN by 5 thousandths
GAP_ENSEMBLE  =  0.050   # within 5 hundredths
CORR_ENSEMBLE =  0.85    # max correlation with any NN anchor

best_nn = max(NN_ANCHORS.values())
verdicts = []
for r in results:
    name = r['name']
    auc  = r['val_macro_auc']
    gap  = best_nn - auc
    if cross is not None and f'classical:{name}' in cross.index:
        max_corr = cross.loc[f'classical:{name}'].max()
    else:
        max_corr = None

    if gap < GAP_REPLACE:
        verdict = 'REPLACE'
        why = f'beats best NN ({best_nn:.4f}) by {-gap:.4f}'
    elif gap < GAP_ENSEMBLE and (max_corr is None or max_corr < CORR_ENSEMBLE):
        verdict = 'ENSEMBLE'
        why = (f'gap {gap:.4f} (< {GAP_ENSEMBLE}); '
               + (f'max corr with NN = {max_corr:.3f} (< {CORR_ENSEMBLE})'
                  if max_corr is not None else 'corr unavailable but gap small'))
    else:
        verdict = 'DROP'
        why = f'gap {gap:.4f}'
        if max_corr is not None:
            why += f', max corr with NN = {max_corr:.3f}'

    verdicts.append({'name': name, 'val_auc': auc, 'gap_vs_best_nn': gap,
                     'max_corr_to_nn': max_corr, 'verdict': verdict, 'reason': why})

v_df = pd.DataFrame(verdicts).sort_values('val_auc', ascending=False)
print(v_df.to_string(index=False))

promote = [v['name'] for v in verdicts if v['verdict'] in ('REPLACE', 'ENSEMBLE')]
if promote:
    print(f'\nPROMOTE to notebook 11 stacker shootout: {promote}')
    print('  Add their OOF predictions (already in `oof_predictions` dict) to the MODELS dict in 11.')
else:
    print('\nNo classical candidate clears the bar. Stay with NN-only ensemble.')


## 7 — How to read the result

**The most likely outcome** at your current ~0.957 NN baseline is `lightgbm` landing somewhere in the 0.85-0.92 
range with high correlation to your NN anchors → `DROP` verdict. That's a useful answer: it confirms the 
candidate selection should stay NN-only and you can stop wondering whether you're leaving easy AUC on the table 
with a tree-based base model.

**The interesting outcome** is `lightgbm` at ~0.88 AUC but with cross-correlation to your NN anchors below 0.85. 
Then it earns a slot in the ensemble *despite* the AUC gap, because its errors are different. Promote to notebook 
11's stacker and check whether the blended val AUC moves up.

**If the GBM beats the NN**, look hard at whether the feature extraction is leaking — e.g., the spec files 
include test-set audio, or the fold split assigns recordings of the same individual bird to both train and val. 
A classical model on summary stats out-scoring a CNN would be a result worth being skeptical of.

**On runtime.** With ~30k clips × 234 species × 4 candidates and `n_jobs=1` per fit, expect 30-90 min total on 
a workstation. The MFCC + per-band-stat extraction is one-shot and cached. If you want to iterate on 
hyperparameters, drop `MAX_FILES_FOR_SMOKE = 5000` in section 0 for a 5-minute pass.

**If the verdict is DROP for all candidates**, the highest-leverage next move is in notebook 11: a stronger 
stacker on top of the NN models you already have. If notebook 11 is also plateauing, that's when it's worth 
flipping `RUN_QUICK_FIT = True` in notebook 10 and putting GPU time into `eca_nfnet_l0`.
